<img src="../../images/bwHPC_Logo_cmyk.svg" width="200" /> <img src="../../images/HochschuleEsslingen_Logo_RGB_DE.png" width="200" /> <img src="../../images/Konstanz_Logo.svg" width="200" /> <img src="../../images/KIT_Logo.png" width="200" />

# Framework Comparison — Exercise

You have now built networks in TensorFlow (`01_TensorFlow`) and in PyTorch
(notebooks 01–04 of this folder). This notebook puts them side by side: **the same
network, on the same data, in both frameworks**, trained with the same
hyperparameters.

The task is a classifier for the **Palmer penguins** — given four body measurements,
decide which of three species a penguin belongs to. The same dataset was used for
k-means in the Machine Learning chapter, but there the species column was hidden.
Here it is the target.

Two things make the comparison worth doing. One is seeing how the same idea reads in
two different styles. The other is subtler: getting two frameworks to actually agree
takes more care than it looks, and this notebook runs into exactly that problem.

Everything that is not specific to a framework — loading the data, encoding the
species, splitting and scaling — is already written for you. What is left blank is
the part that differs between PyTorch and TensorFlow: **building the network,
choosing the optimizer, and running the training**.

A worked version is in
[06_Solution](06_Solution.ipynb). Try it
without looking first.

---

## Contents

1. [The data](#data)
2. [Preparing the data](#prepare)
3. [The network in PyTorch](#pytorch)
4. [The same network in TensorFlow](#tensorflow)
5. [Comparing the two](#compare)
6. [Watching training in TensorBoard](#tensorboard)
7. [References](#references)

<a id="data"></a>
## 1. The data

The **Palmer penguins**: 344 birds of three species, measured at the Palmer Station
in Antarctica. Four of the columns are body measurements, and those are the only
input the network gets.

| Column | Meaning |
|---|---|
| **bill_length_mm** | Length of the bill, in millimetres |
| **bill_depth_mm** | Depth of the bill, in millimetres |
| **flipper_length_mm** | Length of the flipper, in millimetres |
| **body_mass_g** | Body mass, in grams |
| **species** | Adelie, Chinstrap or Gentoo — the target |

Rows with missing values are dropped outright. There are few of them, and every
framework below would otherwise need its own handling.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from palmerpenguins import load_penguins

FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

penguins = load_penguins().dropna(how="any", ignore_index=True)

print("penguins:", len(penguins))
penguins.head()

Before building anything, it is worth seeing whether the four measurements separate
the species at all. Plotting every pair of them, coloured by species, answers that
in one figure.

In [ ]:
pairs = [(a, b) for i, a in enumerate(FEATURES) for b in FEATURES[i + 1:]]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, (xk, yk) in zip(axes.ravel(), pairs):
    for name, group in penguins.groupby("species"):
        ax.scatter(group[xk], group[yk], label=name, alpha=0.6)
    ax.set_xlabel(xk)
    ax.set_ylabel(yk)

axes[0, 0].legend()
fig.tight_layout()
plt.show()

The three species form largely separate clouds in most of those panels, which is
the encouraging answer: a network should be able to learn this.

`species` is not the only thing the dataset could be asked about — `sex` and
`island` are in there too, and the same plot grouped by either of those is worth a
look if you are curious.

<a id="prepare"></a>
## 2. Preparing the data

Three steps, none of them framework-specific.

**The species have to become numbers.** Networks work with numbers, not strings, so
each species is mapped to an index — and the mapping is kept, because the model's
output will be an index that has to be read back.

In [ ]:
species_names = sorted(penguins["species"].unique())
species_to_index = {name: i for i, name in enumerate(species_names)}

X = penguins[FEATURES].to_numpy()
y = penguins["species"].map(species_to_index).to_numpy()

print("mapping:", species_to_index)
print("X:", X.shape, "  y:", y.shape)

**The data is split three ways** — training, validation and test. `stratify` keeps
the proportions of the three species the same in each part, which matters here
because the species are not equally common.

In [ ]:
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.1, random_state=0, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=0, stratify=y_temp
)

print("training:  ", len(X_train))
print("validation:", len(X_valid))
print("test:      ", len(X_test))

**The features have to be scaled.** Body mass runs into the thousands of grams while
bill depth is under 22 millimetres, and a network would take the larger numbers more
seriously purely because they are larger — the same problem, and the same fix, as in
the classification notebook of the TensorFlow chapter.

The scaler is fitted on the training data only.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)

X_train = scaler.transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

print("mean of each scaled training column:", X_train.mean(axis=0).round(3))

Finally the hyperparameters. **Both frameworks get exactly these**, which is the
whole point of the comparison.

In [ ]:
lr = 5e-4         # learning rate
bs = 40           # batch size
epochs = 120
momentum = 0.9

<a id="pytorch"></a>
## 3. The network in PyTorch

The architecture is **4 → 16 → 8 → 3**: four measurements in, three species out, with
two hidden layers and ReLU between them. The last layer has no activation — it
produces raw scores, and the loss function applies the softmax itself.

First the data has to become tensors, wrapped in a `TensorDataset` and handed to a
`DataLoader`, exactly as in [notebook 03](03_Refactoring_With_torch_nn.ipynb).

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train))
valid_ds = TensorDataset(torch.tensor(X_valid, dtype=torch.float32), torch.tensor(y_valid))

train_dl = DataLoader(train_ds, batch_size=bs, shuffle=True)
valid_dl = DataLoader(valid_ds, batch_size=bs * 2)

print("batches per epoch:", len(train_dl))

Now the model itself.

Note that no subclass of `nn.Module` is needed here. `nn.Sequential` already applies
its layers in order and already has a `forward`, so writing a class that does nothing
else would be extra code for nothing. A subclass earns its place when `forward` has
to do something a plain list cannot express — as in the CNN of
[notebook 04](04_Convolutional_Neural_Networks.ipynb), which reshapes its input.

In [ ]:
torch.manual_seed(0)

# Build the network with nn.Sequential: 4 -> 16 -> 8 -> 3,
# with nn.ReLU() between the linear layers and none after the last one.
pt_model = ...

# The optimizer: stochastic gradient descent over pt_model.parameters(),
# using the learning rate and momentum defined above.
opt = ...

pt_model

### The loss, and a trap

`F.cross_entropy` is the loss, as in the previous notebooks. But look at the
`reduction` argument below, because it is the single thing most likely to make this
comparison meaningless.

By default both frameworks **average** the loss over a batch. Passing
`reduction="sum"` makes them add it up instead — which multiplies every gradient by
the batch size, and so multiplies the effective learning rate by 40.

Neither choice is wrong. What matters is that **both frameworks make the same one**.
We use `sum` here, and will use `sum` in TensorFlow too.

In [ ]:
def loss_batch(model, xb, yb, opt=None):
    loss = F.cross_entropy(model(xb), yb, reduction="sum")

    if opt is not None:
        loss.backward()
        opt.step()
        opt.zero_grad()

    return loss.item(), len(xb)

The training loop is the one from notebook 03, with one addition: a
**`SummaryWriter`**, which records the losses to a file that TensorBoard can read
later. `add_scalar` is all it takes — a name, a value, and which epoch it belongs
to.

In [ ]:
def fit(epochs, model, opt, train_dl, valid_dl, writer):
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:
            train_loss, _ = loss_batch(model, xb, yb, opt)

        model.eval()
        with torch.no_grad():
            losses, nums = zip(*[loss_batch(model, xb, yb) for xb, yb in valid_dl])
        valid_loss = np.sum(np.multiply(losses, nums)) / np.sum(nums)

        writer.add_scalar("Loss/train", train_loss, epoch)
        writer.add_scalar("Loss/valid", valid_loss, epoch)

        if epoch % 20 == 19:
            print(f"epoch {epoch + 1:>4}   validation loss {valid_loss:.4f}")

In [ ]:
pt_writer = SummaryWriter(log_dir="./output/logs/PyTorch")

# Train the model: call fit() with the epochs, model, optimizer,
# the two data loaders and pt_writer.
...

pt_writer.close()

### How well did it do?

The model outputs three scores per penguin; the highest one is its answer.

In [ ]:
def pt_accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        predictions = model(torch.tensor(X, dtype=torch.float32)).argmax(dim=1)
    return (predictions.numpy() == y).mean()


pt_test_accuracy = pt_accuracy(pt_model, X_test, y_test)

print(f"training accuracy: {pt_accuracy(pt_model, X_train, y_train):.3f}")
print(f"test accuracy:     {pt_test_accuracy:.3f}")

<a id="tensorflow"></a>
## 4. The same network in TensorFlow

Now the identical model in Keras. Compare this section with the previous one as you
go — the same six ideas appear in both, in a different order and with far less of the
loop written out.

Three things to keep matched with the PyTorch side:

- the same architecture, **4 → 16 → 8 → 3** with ReLU and no final activation
- the same optimizer, **SGD with the same learning rate and momentum**
- the same loss reduction, **`sum`**

`from_logits=True` is the Keras equivalent of leaving the softmax to the loss
function, which is what `F.cross_entropy` did above.

In [ ]:
import tensorflow as tf

tf.keras.utils.set_random_seed(0)

In [ ]:
# Build the same network with tf.keras.Sequential:
#   an Input layer of shape (4,), then Dense(16) and Dense(8) with relu,
#   then Dense(3) with no activation.
tf_model = ...

# Compile it with:
#   optimizer = SGD, same learning rate and momentum as above
#   loss      = SparseCategoricalCrossentropy(from_logits=True, reduction="sum")
#   metrics   = accuracy
...

tf_model.summary()

TensorBoard works differently here. Instead of writing to it by hand inside the
loop, Keras takes a **callback** and does the logging itself — the same information,
none of the code.

In [ ]:
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir="./output/logs/TensorFlow")

# Train the model with fit(): pass X_train and y_train, the validation data,
# the batch size, the number of epochs, and the callback above.
history = ...

print("training finished")

In [ ]:
tf_test_loss, tf_test_accuracy = tf_model.evaluate(X_test, y_test, verbose=0)

print(f"training accuracy: {tf_model.evaluate(X_train, y_train, verbose=0)[1]:.3f}")
print(f"test accuracy:     {tf_test_accuracy:.3f}")

<a id="compare"></a>
## 5. Comparing the two

In [ ]:
print(f"PyTorch    test accuracy: {pt_test_accuracy:.3f}")
print(f"TensorFlow test accuracy: {tf_test_accuracy:.3f}")

The two land in essentially the same place, which is the result we were after: given
the same architecture, the same hyperparameters and the same loss reduction, the
choice of framework does not change what the model learns.

That agreement is more fragile than it looks. Drop `reduction="sum"` from both sides
and re-run, and the two come apart — not because either framework is wrong, but
because the same learning rate then means something different on each side. Defaults
that differ are the usual reason two "identical" implementations disagree.

A confusion matrix shows *where* the mistakes fall, and both models make much the
same ones.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

pt_pred = pt_model(torch.tensor(X_test, dtype=torch.float32)).argmax(dim=1).detach().numpy()
tf_pred = tf_model.predict(X_test, verbose=0).argmax(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, pred, name in [(axes[0], pt_pred, "PyTorch"), (axes[1], tf_pred, "TensorFlow")]:
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, display_labels=species_names, colorbar=False, ax=ax
    )
    ax.set_title(name)

fig.tight_layout()
plt.show()

### Which framework, then?

Neither is better here, and on a problem this size neither ever will be. What the
two sections above actually show is a difference in **style**:

- **TensorFlow / Keras** got the training down to `compile()` and `fit()`. Less to
  write, less to get wrong, and the TensorBoard logging came free with a callback.
- **PyTorch** needed an explicit loop — but that loop is ordinary Python, and
  anything you want to do differently, you simply do.

Which matters depends on whether your model fits the shape the framework expects. For
a standard classifier, Keras is hard to beat. For something unusual, being able to
open the loop is worth a great deal.

<a id="tensorboard"></a>
## 6. Watching training in TensorBoard

Both runs wrote their losses into `./output/logs`, one subdirectory each.
**TensorBoard** reads that directory and draws the curves — and because both
frameworks wrote to the same place, it shows them together.

TensorBoard is a TensorFlow project, so the Keras integration is the deeper of the
two: expect more graphs on the TensorFlow side, including the model structure and
histograms of the weights. The PyTorch run contributes exactly what we asked for with
`add_scalar`, and nothing more.

Two magic commands are all it takes.

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir ./output/logs

> The dashboard is served by *this* notebook's kernel. Restarting the kernel — to
> re-run the training, for instance — takes TensorBoard down with it, and this cell
> has to be run again. That is the one argument for keeping the viewer in a separate
> notebook; for a single pass through the material it is simpler to have it here.

If the panel above stays blank, check that `./output/logs` contains the two run
directories.

In [ ]:
import os

for entry in sorted(os.listdir("./output/logs")):
    print(entry)

<a id="references"></a>
# References

The content of this workshop is in parts based on and inspired by the following sources:

* Python Course of the AG Peter (Prof. Dr. Christine Peter, Kevin Savade, Dr. Oleksandra Kukharenko, Dr. Andrej Berg)
* Software Carpentry workshops (https://software-carpentry.org/lessons/)
* Online KI Kurs des Bundeswettbewerbs Künstliche Intelligenz (https://ki-kurs.org/)
* Real Python (https://realpython.com/)
* Intro to Autoencoders (https://www.tensorflow.org/tutorials/generative/autoencoder)
* Image classification of MNIST using TensorFlow (https://www.kaggle.com/code/viratkothari/image-classification-of-mnist-using-tensorflow)
* The bwHPC wiki (https://wiki.bwhpc.de/)